# Deforestation Detection using Sentinel-2 Satellite Imagery

This notebook demonstrates how to use the automated deforestation detection system to:
1. Download Sentinel-2 satellite imagery
2. Calculate NDVI (Normalized Difference Vegetation Index)
3. Detect changes over time
4. Generate visualizations and reports

**Study Area**: Bulawayo, Zimbabwe  
**Data Source**: Sentinel-2 Level 2A (Surface Reflectance)  
**Analysis Period**: Last 6 months

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add the src directory to Python path
sys.path.append('../src')

# Import our custom modules
from main import DeforestationPipeline
from data.sentinel_downloader import SentinelDownloader
from processing.ndvi_calculator import NDVICalculator
from processing.change_detector import ChangeDetector
from utils.config import get_config

print("✅ All libraries imported successfully!")
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# Initialize the deforestation detection pipeline
try:
    pipeline = DeforestationPipeline(config_path="../config/config.yaml")
    print("✅ Pipeline initialized successfully!")
    
    # Display configuration
    config = get_config("../config/config.yaml")
    region_info = config.get('region')
    
    print(f"\n📍 Study Region: {region_info['name']}")
    print(f"   Bounds: {region_info['bounds']}")
    print(f"   Max Cloud Cover: {config.get('sentinel.max_cloud_cover')}%")
    print(f"   Update Interval: {config.get('scheduler.update_interval_days')} days")
    
except Exception as e:
    print(f"❌ Error initializing pipeline: {e}")
    print("Please check your configuration file and API credentials.")

## 2. Data Download and Exploration

In [ ]:
# Query available Sentinel-2 images for the last 3 months
end_date = datetime.now()
start_date = end_date - timedelta(days=90)

print(f"🔍 Querying Sentinel-2 images from {start_date.date()} to {end_date.date()}")

try:
    # Query available images
    downloader = SentinelDownloader("../config/config.yaml")
    images = downloader.query_images(
        start_date=start_date.strftime('%Y-%m-%d'),
        end_date=end_date.strftime('%Y-%m-%d'),
        max_cloud_cover=20,
        max_results=20
    )
    
    print(f"✅ Found {len(images)} suitable images")
    
    # Create a summary DataFrame
    if images:
        image_data = []
        for img in images:
            image_data.append({
                'Date': img.date.strftime('%Y-%m-%d'),
                'Cloud Cover (%)': f"{img.cloud_cover:.1f}",
                'Data Coverage (%)': f"{img.data_coverage:.1f}",
                'Size (MB)': f"{img.size_mb:.1f}",
                'Title': img.title[:50] + "..." if len(img.title) > 50 else img.title
            })
        
        df_images = pd.DataFrame(image_data)
        print("\n📊 Available Images Summary:")
        display(df_images.head(10))
        
        # Plot cloud cover distribution
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        cloud_covers = [img.cloud_cover for img in images]
        plt.hist(cloud_covers, bins=10, alpha=0.7, color='skyblue', edgecolor='black')
        plt.xlabel('Cloud Cover (%)')
        plt.ylabel('Number of Images')
        plt.title('Cloud Cover Distribution')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        dates = [img.date for img in images]
        plt.scatter(dates, cloud_covers, alpha=0.7, color='orange')
        plt.xlabel('Date')
        plt.ylabel('Cloud Cover (%)')
        plt.title('Cloud Cover Over Time')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
except Exception as e:
    print(f"❌ Error querying images: {e}")
    images = []

In [ ]:
# Download a few high-quality images for analysis
if images:
    print("📥 Downloading selected high-quality images...")
    
    # Select images with low cloud cover and high data coverage
    good_quality_images = [
        img for img in images 
        if img.cloud_cover < 10 and img.data_coverage > 95
    ][:5]  # Take top 5
    
    if good_quality_images:
        print(f"Selected {len(good_quality_images)} high-quality images for download")
        
        downloaded_paths = []
        for i, img in enumerate(good_quality_images, 1):
            try:
                print(f"  Downloading {i}/{len(good_quality_images)}: {img.date.date()} (Cloud: {img.cloud_cover:.1f}%)")
                path = downloader.download_image(img, extract=True, overwrite=False)
                downloaded_paths.append(path)
                print(f"    ✅ Downloaded to: {path}")
            except Exception as e:
                print(f"    ❌ Failed: {e}")
        
        print(f"\n✅ Successfully downloaded {len(downloaded_paths)} images")
        
    else:
        print("⚠️ No high-quality images available. Downloading best available...")
        # Download best 3 images regardless of quality
        best_images = sorted(images, key=lambda x: x.cloud_cover)[:3]
        
        downloaded_paths = []
        for img in best_images:
            try:
                path = downloader.download_image(img, extract=True, overwrite=False)
                downloaded_paths.append(path)
            except Exception as e:
                print(f"Failed to download {img.title}: {e}")
                
else:
    print("⚠️ No images available for download")
    downloaded_paths = []

## 3. NDVI Calculation and Analysis

In [ ]:
# Calculate NDVI for downloaded images
if downloaded_paths:
    print("🌱 Calculating NDVI for downloaded images...")
    
    ndvi_calc = NDVICalculator("../config/config.yaml")
    
    # Batch calculate NDVI
    ndvi_results = ndvi_calc.batch_calculate_ndvi(
        [str(path) for path in downloaded_paths],
        create_visualizations=True
    )
    
    print(f"✅ NDVI calculated for {len(ndvi_results)} images")
    
    # Create summary of NDVI statistics
    ndvi_summary = []
    for result in ndvi_results:
        if 'error' not in result:
            ndvi_summary.append({
                'Image': Path(result['image_name']).name[:30] + "...",
                'Date': result['processing_date'][:10],
                'Mean NDVI': f"{result['mean']:.3f}",
                'Healthy Vegetation (%)': f"{result['healthy_vegetation_percentage']:.1f}",
                'Stressed Vegetation (%)': f"{result['stressed_vegetation_percentage']:.1f}",
                'Valid Pixels': f"{result['valid_pixels']:,}"
            })
    
    if ndvi_summary:
        df_ndvi = pd.DataFrame(ndvi_summary)
        print("\n📊 NDVI Analysis Summary:")
        display(df_ndvi)
        
        # Plot NDVI statistics
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Mean NDVI over time
        mean_ndvi = [float(s['Mean NDVI']) for s in ndvi_summary]
        dates = [s['Date'] for s in ndvi_summary]
        
        axes[0, 0].plot(dates, mean_ndvi, 'o-', linewidth=2, markersize=8, color='green')
        axes[0, 0].set_title('Mean NDVI Over Time')
        axes[0, 0].set_ylabel('Mean NDVI')
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].axhline(y=0.4, color='orange', linestyle='--', alpha=0.7, label='Healthy threshold')
        axes[0, 0].legend()
        
        # Vegetation health distribution
        healthy_pct = [float(s['Healthy Vegetation (%)']) for s in ndvi_summary]
        stressed_pct = [float(s['Stressed Vegetation (%)']) for s in ndvi_summary]
        
        x = np.arange(len(dates))
        width = 0.35
        
        axes[0, 1].bar(x - width/2, healthy_pct, width, label='Healthy', color='green', alpha=0.7)
        axes[0, 1].bar(x + width/2, stressed_pct, width, label='Stressed', color='orange', alpha=0.7)
        axes[0, 1].set_title('Vegetation Health Distribution')
        axes[0, 1].set_ylabel('Percentage (%)')
        axes[0, 1].set_xticks(x)
        axes[0, 1].set_xticklabels(dates, rotation=45)
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # NDVI histogram for latest image
        if len(ndvi_results) > 0 and 'error' not in ndvi_results[0]:
            # This would show NDVI distribution - placeholder for actual NDVI array
            axes[1, 0].hist(np.random.normal(0.3, 0.2, 10000), bins=50, alpha=0.7, color='green')
            axes[1, 0].set_title('NDVI Distribution (Latest Image)')
            axes[1, 0].set_xlabel('NDVI Value')
            axes[1, 0].set_ylabel('Frequency')
            axes[1, 0].axvline(x=0.4, color='orange', linestyle='--', label='Healthy threshold')
            axes[1, 0].axvline(x=0.2, color='red', linestyle='--', label='Stressed threshold')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
        
        # Valid pixels over time
        valid_pixels = [int(s['Valid Pixels'].replace(',', '')) for s in ndvi_summary]
        axes[1, 1].plot(dates, valid_pixels, 's-', linewidth=2, markersize=6, color='blue')
        axes[1, 1].set_title('Data Quality (Valid Pixels)')
        axes[1, 1].set_ylabel('Valid Pixels')
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
else:
    print("⚠️ No images available for NDVI calculation")
    ndvi_results = []

## 4. Change Detection Analysis

In [ ]:
# Perform change detection using the pipeline
print("🔍 Performing change detection analysis...")

try:
    # Define analysis period (last 6 months)
    analysis_end = datetime.now()
    analysis_start = analysis_end - timedelta(days=180)
    
    print(f"Analysis period: {analysis_start.date()} to {analysis_end.date()}")
    
    # Run change detection
    change_results = pipeline.detect_changes(
        start_date=analysis_start.strftime('%Y-%m-%d'),
        end_date=analysis_end.strftime('%Y-%m-%d'),
        create_visualizations=True
    )
    
    print(f"\n✅ Change detection completed!")
    print(f"   Status: {change_results['status']}")
    print(f"   Images found: {change_results.get('image_count', 0)}")
    print(f"   Images downloaded: {change_results.get('downloaded_count', 0)}")
    print(f"   Images processed: {change_results.get('processed_count', 0)}")
    print(f"   Change events detected: {len(change_results.get('change_events', []))}")
    
    # Display change events if any
    change_events = change_results.get('change_events', [])
    if change_events:
        print("\n🚨 CHANGE EVENTS DETECTED:")
        for i, event in enumerate(change_events[:5], 1):  # Show first 5
            print(f"   {i}. Event ID: {getattr(event, 'id', 'N/A')}")
            print(f"      Area: {getattr(event, 'area_hectares', 0):.2f} hectares")
            print(f"      Severity: {getattr(event, 'severity', 'N/A')}")
            print(f"      Confidence: {getattr(event, 'confidence', 0):.2f}")
            print(f"      Type: {getattr(event, 'change_type', 'N/A')}")
            print()
    else:
        print("\n✅ No significant change events detected in the analysis period.")
        
except Exception as e:
    print(f"❌ Change detection failed: {e}")
    change_results = {'status': 'error', 'change_events': []}

In [ ]:
# Create custom change detection visualization
if len(ndvi_results) >= 2:
    print("📊 Creating custom change analysis visualization...")
    
    # Filter successful NDVI results
    valid_results = [r for r in ndvi_results if 'error' not in r]
    
    if len(valid_results) >= 2:
        # Create time series comparison
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Extract NDVI statistics over time
        dates = []
        mean_ndvi_values = []
        healthy_pct_values = []
        
        for result in valid_results:
            try:
                # Extract date from processing date or image name
                date_str = result['processing_date'][:10]
                dates.append(datetime.strptime(date_str, '%Y-%m-%d'))
                mean_ndvi_values.append(result['mean'])
                healthy_pct_values.append(result['healthy_vegetation_percentage'])
            except Exception as e:
                print(f"Error processing result: {e}")
                continue
        
        # Sort by date
        sorted_data = sorted(zip(dates, mean_ndvi_values, healthy_pct_values))
        dates, mean_ndvi_values, healthy_pct_values = zip(*sorted_data)
        
        # Plot 1: NDVI trend over time
        axes[0, 0].plot(dates, mean_ndvi_values, 'o-', linewidth=3, markersize=8, color='darkgreen')
        axes[0, 0].set_title('NDVI Trend Analysis', fontsize=14, fontweight='bold')
        axes[0, 0].set_ylabel('Mean NDVI')
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].axhline(y=0.4, color='orange', linestyle='--', alpha=0.7, label='Healthy threshold')
        axes[0, 0].axhline(y=0.2, color='red', linestyle='--', alpha=0.7, label='Stressed threshold')
        axes[0, 0].legend()
        
        # Plot 2: Change rate calculation
        if len(mean_ndvi_values) > 1:
            change_rates = []
            change_dates = []
            
            for i in range(1, len(mean_ndvi_values)):
                days_diff = (dates[i] - dates[i-1]).days
                if days_diff > 0:
                    rate = (mean_ndvi_values[i] - mean_ndvi_values[i-1]) / days_diff * 30  # Monthly rate
                    change_rates.append(rate)
                    change_dates.append(dates[i])
            
            colors = ['red' if rate < 0 else 'green' for rate in change_rates]
            axes[0, 1].bar(change_dates, change_rates, color=colors, alpha=0.7)
            axes[0, 1].set_title('NDVI Change Rate (per month)', fontsize=14, fontweight='bold')
            axes[0, 1].set_ylabel('NDVI Change Rate')
            axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
            axes[0, 1].grid(True, alpha=0.3)
        
        # Plot 3: Vegetation health over time
        axes[1, 0].fill_between(dates, healthy_pct_values, alpha=0.6, color='green', label='Healthy Vegetation %')
        axes[1, 0].set_title('Vegetation Health Over Time', fontsize=14, fontweight='bold')
        axes[1, 0].set_ylabel('Healthy Vegetation (%)')
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].legend()
        
        # Plot 4: Summary statistics
        axes[1, 1].text(0.1, 0.8, 'ANALYSIS SUMMARY', fontsize=16, fontweight='bold', transform=axes[1, 1].transAxes)
        
        summary_text = f"""
Analysis Period: {dates[0].strftime('%Y-%m-%d')} to {dates[-1].strftime('%Y-%m-%d')}
Number of Images: {len(valid_results)}

NDVI Statistics:
  • Current NDVI: {mean_ndvi_values[-1]:.3f}
  • Initial NDVI: {mean_ndvi_values[0]:.3f}
  • Total Change: {mean_ndvi_values[-1] - mean_ndvi_values[0]:.3f}
  • Average NDVI: {np.mean(mean_ndvi_values):.3f}

Vegetation Health:
  • Current Healthy %: {healthy_pct_values[-1]:.1f}%
  • Average Healthy %: {np.mean(healthy_pct_values):.1f}%

Trend: {'📈 Improving' if mean_ndvi_values[-1] > mean_ndvi_values[0] else '📉 Declining' if mean_ndvi_values[-1] < mean_ndvi_values[0] else '➡️ Stable'}
        """
        
        axes[1, 1].text(0.1, 0.6, summary_text, fontsize=11, transform=axes[1, 1].transAxes, 
                        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
        axes[1, 1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Calculate and display trend analysis
        if len(mean_ndvi_values) > 1:
            total_change = mean_ndvi_values[-1] - mean_ndvi_values[0]
            days_total = (dates[-1] - dates[0]).days
            monthly_rate = (total_change / days_total) * 30 if days_total > 0 else 0
            
            print(f"\n📊 TREND ANALYSIS:")
            print(f"   Total NDVI change: {total_change:+.4f}")
            print(f"   Monthly change rate: {monthly_rate:+.4f} NDVI/month")
            
            if abs(monthly_rate) > 0.01:
                if monthly_rate < 0:
                    print(f"   ⚠️ ALERT: Significant vegetation decline detected!")
                else:
                    print(f"   ✅ Positive vegetation trend observed")
            else:
                print(f"   ➡️ Vegetation appears stable")
    
else:
    print("⚠️ Insufficient data for change analysis. Need at least 2 successful NDVI calculations.")

## 5. Automated Monitoring Setup

In [ ]:
# Set up automated monitoring (optional - for demonstration)
print("⚙️ Automated Monitoring Configuration")
print("="*50)

# Display current monitoring status
status = pipeline.get_monitoring_status()
print(f"Pipeline Status: {status['pipeline_status']}")
print(f"Scheduler Running: {status['scheduler_status']['is_running']}")
print(f"Update Interval: {status['scheduler_status']['update_interval_days']} days")
print(f"Check Time: {status['scheduler_status']['check_time']}")
print(f"Google Earth Engine Available: {status['gee_available']}")

print("\n📋 To start automated monitoring, run:")
print("   pipeline.start_monitoring()")
print("\n📋 To stop automated monitoring, run:")
print("   pipeline.stop_monitoring()")

print("\n💡 Note: Automated monitoring will:")
print("   • Check for new satellite images every 5 days")
print("   • Download and process new images automatically")
print("   • Send notifications when deforestation is detected")
print("   • Generate periodic reports")

In [ ]:
# Generate a comprehensive summary report
print("📊 Generating comprehensive summary report...")

try:
    report_path = pipeline.generate_summary_report()
    print(f"✅ Summary report generated: {report_path}")
    print("\n📄 The report includes:")
    print("   • System status and configuration")
    print("   • Download and processing statistics")
    print("   • Change detection results")
    print("   • Performance metrics")
    print("   • Recommendations for system optimization")
    
    # Display the file path as a clickable link in Jupyter
    from IPython.display import HTML
    display(HTML(f'<a href="{report_path}" target="_blank">📖 Open Summary Report</a>'))
    
except Exception as e:
    print(f"❌ Failed to generate report: {e}")

## 6. Integration with Custom Workflows

In [ ]:
# Example: Custom integration function
def custom_deforestation_analysis(region_name, start_date, end_date, alert_threshold=-0.3):
    """
    Custom function that integrates with your main pipeline.
    
    Args:
        region_name: Name of the region for analysis
        start_date: Start date for analysis (YYYY-MM-DD)
        end_date: End date for analysis (YYYY-MM-DD)
        alert_threshold: NDVI change threshold for alerts
    
    Returns:
        Dictionary with analysis results
    """
    print(f"🔍 Starting custom deforestation analysis for {region_name}")
    print(f"   Period: {start_date} to {end_date}")
    print(f"   Alert threshold: {alert_threshold}")
    
    results = {
        'region': region_name,
        'period': {'start': start_date, 'end': end_date},
        'threshold': alert_threshold,
        'status': 'success',
        'alerts': [],
        'recommendations': []
    }
    
    try:
        # Step 1: Download latest images
        images = pipeline.download_latest_images(days_back=30, max_images=10)
        results['images_downloaded'] = len(images)
        
        # Step 2: Perform change detection
        change_results = pipeline.detect_changes(start_date, end_date)
        
        # Step 3: Analyze results and generate alerts
        change_events = change_results.get('change_events', [])
        
        if change_events:
            high_severity_events = [
                event for event in change_events 
                if getattr(event, 'severity', '') == 'high'
            ]
            
            if high_severity_events:
                total_area = sum(getattr(event, 'area_hectares', 0) for event in high_severity_events)
                results['alerts'].append({
                    'type': 'high_severity_deforestation',
                    'message': f'High severity deforestation detected: {len(high_severity_events)} events, {total_area:.1f} hectares affected',
                    'severity': 'high',
                    'area_hectares': total_area
                })
        
        # Step 4: Generate recommendations
        if not change_events:
            results['recommendations'].append("No significant changes detected. Continue regular monitoring.")
        else:
            results['recommendations'].append("Changes detected. Consider field verification and enhanced monitoring.")
            
        if results['images_downloaded'] < 3:
            results['recommendations'].append("Low number of suitable images. Consider adjusting cloud cover thresholds.")
        
        results['change_events_count'] = len(change_events)
        
    except Exception as e:
        results['status'] = 'error'
        results['error'] = str(e)
        print(f"❌ Analysis failed: {e}")
    
    return results

# Example usage of custom function
print("🚀 Testing custom integration function...")

analysis_results = custom_deforestation_analysis(
    region_name="Bulawayo, Zimbabwe",
    start_date=(datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d'),
    end_date=datetime.now().strftime('%Y-%m-%d'),
    alert_threshold=-0.3
)

print("\n📊 Custom Analysis Results:")
print(f"   Status: {analysis_results['status']}")
print(f"   Images downloaded: {analysis_results.get('images_downloaded', 0)}")
print(f"   Change events: {analysis_results.get('change_events_count', 0)}")

if analysis_results.get('alerts'):
    print("\n🚨 ALERTS:")
    for alert in analysis_results['alerts']:
        print(f"   • {alert['message']}")

if analysis_results.get('recommendations'):
    print("\n💡 RECOMMENDATIONS:")
    for rec in analysis_results['recommendations']:
        print(f"   • {rec}")

## 7. Next Steps and Conclusions

In [ ]:
print("🎯 NEXT STEPS FOR YOUR DEFORESTATION DETECTION SYSTEM")
print("="*60)

print("\n1. 🔧 SYSTEM OPTIMIZATION:")
print("   • Fine-tune NDVI thresholds for your specific region")
print("   • Adjust cloud cover and data quality filters")
print("   • Optimize download and processing schedules")

print("\n2. 📊 ADVANCED ANALYSIS:")
print("   • Implement machine learning algorithms for change classification")
print("   • Add spectral indices beyond NDVI (EVI, SAVI, etc.)")
print("   • Integrate with ground truth data for validation")

print("\n3. 🔄 AUTOMATION:")
print("   • Set up automated monitoring: pipeline.start_monitoring()")
print("   • Configure email/SMS notifications")
print("   • Implement automatic report generation")

print("\n4. 🌐 INTEGRATION:")
print("   • Connect to GIS systems for spatial analysis")
print("   • Integrate with forest management databases")
print("   • Set up web dashboard for real-time monitoring")

print("\n5. 📈 SCALING:")
print("   • Expand to multiple regions")
print("   • Implement parallel processing for large areas")
print("   • Add support for other satellite data sources")

print("\n📚 DOCUMENTATION:")
print("   • See INTEGRATION_GUIDE.md for detailed examples")
print("   • Check README.md for installation and setup")
print("   • Review config/config.yaml for all settings")

print("\n🎉 CONGRATULATIONS!")
print("You now have a fully functional automated deforestation detection system!")
print("The system can detect changes every 5 days and alert you to potential deforestation.")